In [1]:
##############
## INITIATE ##
##############

## Imports
import numpy as np
import os
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

## Import modules
from f_dataset import SegmentationDataset
from f_dataset import augment
# from f_model import UNet as Model
from f_model_transformer import UNetTransformer as Model
from f_train import train 
from f_train import test
from f_train import dice_with_logits_loss

#
###

In [3]:
#################################
## INITIATE MODEL AND DATASETS ##
#################################

## Parameters
pt_dataset_l = '/Users/user/Documents/projects/GlandSeg/data/SEGG/images/train_tiles/'
pt_dataset_v = '/Users/user/Documents/projects/GlandSeg/data/SEGG/images/val_tiles/'
thresholds = np.array([0, 150, 210, 255]) # (Background/stroma, epithelia, lumen)
# thresholds = np.array([0, 150, 255]) # (Background/stroma, epithelia/lumen) 
batch_size = 3
num_epochs = 90
check_save = 1
num_classes = 3
pt_model_in = 'models/TransUNet_V1_0.pth'
pt_model_out = 'models/TransUNet_V1_1.pth'

## Create dataset and dataloader
dataset_l = SegmentationDataset(pt_dataset_l, thresholds, transform=augment)
dataset_v = SegmentationDataset(pt_dataset_v, thresholds, transform=None)
dataloader_l = DataLoader(dataset_l, batch_size=batch_size, shuffle=True)
dataloader_v = DataLoader(dataset_v, batch_size=batch_size, shuffle=False)

## Load model
device = 'mps'
print(f"Using {device} device")
model = Model(n_classes=num_classes).to(device)
if os.path.exists(pt_model_in):
    model.load_state_dict(torch.load(pt_model_in))
    print('Loaded model')
else:
    torch.save(model.state_dict(), pt_model_in)

## Loss
# loss_fn = dice_with_logits_loss
loss_fn = nn.CrossEntropyLoss()

## Optimizer
# optimizer = torch.optim.SGD(model.parameters(), lr=1e-4)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

#
###

Using mps device


In [4]:
#################
## TRAIN MODEL ##
#################

## Training
print('Training starts...')
train_loss = []
val_loss = []
for t in range(num_epochs):
    print(f"-- Epoch {t+1} ------------- \n")
    train_loss += [train(dataloader_l, model, loss_fn, optimizer, device)]
    val_loss += [test(dataloader_v, model, loss_fn, device)]
    if (t + 1) % check_save == 0:
        torch.save(model.state_dict(), pt_model_in.replace('.pth',f"_{t+1}.pth"))
        
## End training
print('Training complete')
torch.save(model.state_dict(), pt_model_out)

#
###

Training starts...
-- Epoch 1 ------------- 

Progress: [    0/ 2371]  Loss: 1.287918
Progress: [   30/ 2371]  Loss: 0.973985
Progress: [   60/ 2371]  Loss: 1.020214
Progress: [   90/ 2371]  Loss: 0.927812
Progress: [  120/ 2371]  Loss: 0.931660
Progress: [  150/ 2371]  Loss: 0.724289
Progress: [  180/ 2371]  Loss: 1.024398
Progress: [  210/ 2371]  Loss: 1.023451
Progress: [  240/ 2371]  Loss: 0.883761
Progress: [  270/ 2371]  Loss: 0.877173
Progress: [  300/ 2371]  Loss: 0.792985
Progress: [  330/ 2371]  Loss: 0.669124
Progress: [  360/ 2371]  Loss: 0.634521
Progress: [  390/ 2371]  Loss: 0.695067
Progress: [  420/ 2371]  Loss: 0.648554
Progress: [  450/ 2371]  Loss: 0.975186
Progress: [  480/ 2371]  Loss: 0.607958
Progress: [  510/ 2371]  Loss: 0.844914
Progress: [  540/ 2371]  Loss: 0.718432
Progress: [  570/ 2371]  Loss: 0.740324
Progress: [  600/ 2371]  Loss: 0.559479
Progress: [  630/ 2371]  Loss: 0.916506
Progress: [  660/ 2371]  Loss: 0.816867
Progress: [  690/ 2371]  Loss: 0.4